In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd()
while not (RAIZ / 'src').is_dir() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.database.db import get_engine
from dotenv import load_dotenv

load_dotenv(RAIZ / '.env')
engine = get_engine()

df = pd.read_sql('SELECT * FROM staging.indicadores_macro ORDER BY data', engine)
df['data'] = pd.to_datetime(df['data'])
df = df.set_index('data')

print(f'{df.shape[0]} meses | {df.index.min():%b/%Y} → {df.index.max():%b/%Y}')
df.tail(3)

In [ ]:
# shift total = lag econômico (bloco 7) + atraso de publicação
feat = pd.DataFrame({
    'ipca':          df['ipca_pct'],                            # alvo
    'd_selic':       df['taxa_selic'].diff().shift(1),          # Copom divulga no mês
    'd_cambio':      df['cambio_usd'].pct_change().shift(4),    # diário, sem atraso
    'd_ibcbr':       df['ibcbr'].pct_change().shift(3),         # 1 econ + 2 publicação
    'd_focus':       df['ipca_focus'].diff().shift(5),          # semanal, quase tempo real
    'desemprego':    df['desemprego'].shift(2),                 # PNAD-C ~1m de atraso
    'mes':           df.index.month,                            # sazonalidade
})
CANDIDATAS = ['d_selic', 'd_cambio', 'd_ibcbr', 'd_focus', 'desemprego']

# alvo = IPCA do MÊS SEGUINTE (previsão de 1 passo à frente)
feat['y'] = feat['ipca'].shift(-1)

feat = feat.dropna(subset=['y'])
print(feat.tail())

In [ ]:
def rolling_origin_naive(feat: pd.DataFrame, meses_teste: int = 24):
    """Prevê o IPCA do mês seguinte usando o último valor conhecido."""
    feat = feat.sort_index().dropna(subset=['y', 'ipca'])

    if len(feat) <= meses_teste:
        raise ValueError("Dados insuficientes para treino e teste.")

    erros = []

    for i in range(len(feat) - meses_teste, len(feat)):
        treino = feat.iloc[:i]
        real = feat['y'].iloc[i]
        previsto = treino['ipca'].iloc[-1]
        erros.append(abs(real - previsto))

    return np.mean(erros)


mae_naive = rolling_origin_naive(feat)
print(f"Naive: MAE = {mae_naive:.3f} p.p.")

In [ ]:
def rolling_origin_media_movel(feat: pd.DataFrame, meses_teste: int = 24, janela: int = 12):
    erros = []
    for i in range(len(feat) - meses_teste, len(feat)):
        treino = feat.iloc[:i]
        real = feat['y'].iloc[i]
        previsto = treino['ipca'].tail(janela).mean()
        erros.append(abs(real - previsto))
    return np.mean(erros)

mae_mm = rolling_origin_media_movel(feat)
print(f'Média móvel 12m: MAE = {mae_mm:.3f} p.p.')

In [ ]:
from sklearn.linear_model import Ridge

def seleciona_features(treino: pd.DataFrame, candidatas: list, k: int = 3) -> list:
    """SÓ olha o treino do fold — nunca o teste."""
    corrs = {c: abs(treino[c].corr(treino['ipca'])) for c in candidatas}
    return sorted(corrs, key=corrs.get, reverse=True)[:k]

def rolling_origin_ridge(feat: pd.DataFrame, meses_teste: int = 24,
                         k: int = 3, alpha: float = 1.0) -> float:
    erros = []
    for i in range(len(feat) - meses_teste, len(feat)):
        treino = feat.iloc[:i]
        real = feat['y'].iloc[i]

        cols = seleciona_features(treino, CANDIDATAS, k)
        treino_m = treino.dropna(subset=cols + ['ipca'])
        x_hoje = feat.iloc[[i]][cols]

        if treino_m.empty or x_hoje.isna().any(axis=None):
            previsto = treino['ipca'].iloc[-1]     # fallback: vira naive
        else:
            modelo = Ridge(alpha=alpha)
            modelo.fit(treino_m[cols], treino_m['ipca'])
            previsto = modelo.predict(x_hoje)[0]

        erros.append(abs(real - previsto))
    return np.mean(erros)

mae_ridge = rolling_origin_ridge(feat)
print(f'Ridge (top-{3} por fold): MAE = {mae_ridge:.3f} p.p.')

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings("ignore")   # statsmodels é prolixo

def rolling_origin_arima(feat: pd.DataFrame, meses_teste: int = 24,
                         order: tuple = (1, 1, 1)) -> float:
    erros = []
    for i in range(len(feat) - meses_teste, len(feat)):
        treino = feat.iloc[:i]
        real = feat['y'].iloc[i]
        try:
            modelo = ARIMA(treino['ipca'], order=order).fit()
            previsto = modelo.forecast(steps=1).iloc[0]
        except Exception:
            previsto = treino['ipca'].iloc[-1]     # fallback naive
        erros.append(abs(real - previsto))
    return np.mean(erros)

mae_arima = rolling_origin_arima(feat)
arima_order = (1, 1, 1)
print(f'ARIMA{arima_order}: MAE = {mae_arima:.3f} p.p.')

Resultados de previsão do IPCA — horizonte de 1 mês

No horizonte de previsão de um mês, avaliado entre outubro de 2024 e outubro de 2026, a média móvel de 12 meses apresentou o menor erro absoluto médio (MAE), de 0,279 p.p., seguida pelo ARIMA(1,1,1), com 0,297 p.p., e pelo Ridge com variáveis macroeconômicas, com 0,304 p.p. Todos superaram o modelo naive, cujo MAE foi de 0,351 p.p.

Os resultados sugerem que o histórico recente do IPCA constitui uma referência preditiva competitiva no curto prazo. Embora o Ridge tenha superado o naive, ainda não há evidência suficiente para concluir que as variáveis macroeconômicas proporcionam ganhos consistentes em relação à média móvel. Essa hipótese deve ser investigada com validação temporal rigorosa.

Conclusão: para previsões pontuais de um mês à frente, a média móvel de 12 meses é o melhor modelo entre os avaliados. Como próximas linhas de investigação (hipóteses v2), ficam a previsão do IPCA acumulado em 12 meses e o nowcasting, que poderão explorar relações econômicas não capturadas pelos modelos atuais.